# Limpieza del dataset de hospitales

Este notebook transforma el CSV original, que trae una fila por especialidad, en una tabla por hospital pensada para el prototipo de ambulancias.

Objetivos:
- conservar las especialidades de cada hospital en una sola fila
- normalizar nombres de columnas y valores vacíos
- generar coordenadas útiles para geolocalización
- añadir campos que ayuden a filtrar hospitales por síntomas o gravedad

In [1]:
from pathlib import Path

import pandas as pd
from pyproj import Transformer

NOTEBOOK_CWD = Path.cwd().resolve()
RAW_FILENAME = "centros_servicios_establecimientos_sanitarios.csv"

PROJECT_DIR = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD / "analisis_datos", NOTEBOOK_CWD, NOTEBOOK_CWD.parent]
        if (candidate / "data" / "raw" / RAW_FILENAME).exists()
    ),
    NOTEBOOK_CWD,
)

RAW_PATH = PROJECT_DIR / "data" / "raw" / RAW_FILENAME
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_PATH = PROCESSED_DIR / "centros_servicios_establecimientos_sanitarios_limpio.csv"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

Antes de iniciar, es importante revisar el dataset original para entender su estructura y contenido. Esto nos permitirá identificar las columnas relevantes y planificar cómo consolidar la información de las especialidades en una sola fila por hospital.

In [7]:
data = pd.read_csv(RAW_PATH, sep=";", dtype=str, keep_default_na=False)
data.head()

,centro_nro_registro,centro_tipo,dependecia_funcional,dependecia_patrimonial,oferta_asistecial,municipio_nombre,direccion_vial_tipo,direccion_vial_nombre,direccion_vial_nro,direccion_informacon_adicional,direccion_codigo_postal,localizacion_coordenada_x,localizacion_coordenada_y
0,CH0001,Hospital especializado,Mutua de Accidentes de Trabajo,Mútua Patronal,Obtención de muestras,Majadahonda,CTRA,M-515 (POZUELO),61,EDIFICIO A-C-D,28220,427057,4478720
1,CH0001,Hospital especializado,Mutua de Accidentes de Trabajo,Mútua Patronal,Otras unidades asistenciales,Majadahonda,CTRA,M-515 (POZUELO),61,EDIFICIO A-C-D,28220,427057,4478720
2,CH0001,Hospital especializado,Mutua de Accidentes de Trabajo,Mútua Patronal,Urología,Majadahonda,CTRA,M-515 (POZUELO),61,EDIFICIO A-C-D,28220,427057,4478720
3,CH0001,Hospital especializado,Mutua de Accidentes de Trabajo,Mútua Patronal,Radiodiagnóstico,Majadahonda,CTRA,M-515 (POZUELO),61,EDIFICIO A-C-D,28220,427057,4478720
4,CH0001,Hospital especializado,Mutua de Accidentes de Trabajo,Mútua Patronal,Atención Continuada en Atención Primaria,Majadahonda,CTRA,M-515 (POZUELO),61,EDIFICIO A-C-D,28220,427057,4478720


In [3]:
import unicodedata


def clean_value(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "null"}:
        return pd.NA
    return text


def normalize_text(text):
    if pd.isna(text):
        return ""
    normalized = unicodedata.normalize("NFKD", str(text))
    normalized = normalized.encode("ascii", "ignore").decode("ascii")
    return " ".join(normalized.lower().split())


def first_non_null(series):
    valid = series.dropna()
    return valid.iloc[0] if not valid.empty else pd.NA


def join_unique(series, separator=" | "):
    values = []
    for value in series:
        cleaned = clean_value(value)
        if pd.isna(cleaned):
            continue
        if cleaned not in values:
            values.append(cleaned)
    return separator.join(values)


def build_attention_profiles(especialidades_series):
    normalized_specialties = {normalize_text(value) for value in especialidades_series.dropna()}
    profiles = []

    profile_keywords = {
        "cardiovascular": {
            "cardiologia",
            "cirugia cardiaca",
            "angiologia y cirugia vascular",
        },
        "neurologia": {
            "neurologia",
            "neurocirugia",
            "neurofisiologia",
        },
        "trauma_ortopedia": {
            "cirugia ortopedica y traumatologia",
            "lesionados medulares",
        },
        "obstetricia_neonatal": {
            "enfermeria obstetrico-ginecologica (matrona)",
            "cuidados intensivos neonatales",
            "cuidados intermedios neonatales",
            "fecundacion in vitro",
            "ginecologia",
            "obstetricia",
        },
        "pediatria": {
            "cirugia pediatrica",
            "pediatria",
            "cuidados intensivos neonatales",
            "cuidados intermedios neonatales",
        },
        "salud_mental": {
            "psiquiatria",
        },
        "diagnostico": {
            "radiodiagnostico",
            "laboratorio clinico",
            "bioquimica clinica",
            "anatomia patologica",
        },
        "urgencias_criticas": {
            "anestesia y reanimacion",
            "atencion continuada en atencion primaria",
            "cuidados intensivos",
        },
        "rehabilitacion": {
            "fisioterapia",
            "rehabilitacion",
            "logopedia",
            "terapia ocupacional",
        },
        "cirugia": {
            "cirugia general y digestivo",
            "cirugia mayor ambulatoria",
            "cirugia menor ambulatoria",
            "cirugia maxilofacial",
            "cirugia plastica y reparadora",
            "cirugia toracica",
            "cirugia refractiva",
        },
        "oncologia": {
            "oncologia",
            "hematologia y hemoterapia",
        },
    }

    for profile_name, keywords in profile_keywords.items():
        if normalized_specialties.intersection(keywords):
            profiles.append(profile_name)

    return " | ".join(profiles)


column_renames = {
    "centro_nro_registro": "centro_id",
    "centro_tipo": "centro_tipo",
    "dependecia_funcional": "dependencia_funcional",
    "dependecia_patrimonial": "dependencia_patrimonial",
    "oferta_asistecial": "especialidad",
    "municipio_nombre": "municipio",
    "direccion_vial_tipo": "tipo_via",
    "direccion_vial_nombre": "nombre_via",
    "direccion_vial_nro": "numero_via",
    "direccion_informacon_adicional": "info_adicional",
    "direccion_codigo_postal": "codigo_postal",
    "localizacion_coordenada_x": "utm_x",
    "localizacion_coordenada_y": "utm_y",
}

raw = pd.read_csv(RAW_PATH, sep=";", dtype=str, keep_default_na=False)
df = raw.rename(columns=column_renames).copy()
df = df.apply(lambda column: column.map(clean_value))

for column in ["utm_x", "utm_y"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

transformer = Transformer.from_crs(25830, 4326, always_xy=True)
valid_coordinates = df["utm_x"].notna() & df["utm_y"].notna()
longitudes, latitudes = transformer.transform(
    df.loc[valid_coordinates, "utm_x"].to_numpy(),
    df.loc[valid_coordinates, "utm_y"].to_numpy(),
)
df.loc[valid_coordinates, "lon"] = longitudes
df.loc[valid_coordinates, "lat"] = latitudes

df["direccion_completa"] = (
    df[["tipo_via", "nombre_via", "numero_via", "info_adicional", "municipio", "codigo_postal"]]
    .fillna("")
    .agg(lambda row: " ".join(value for value in row if value), axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

cleaned = (
    df.groupby("centro_id", as_index=False)
    .agg(
        centro_tipo=("centro_tipo", first_non_null),
        dependencia_funcional=("dependencia_funcional", first_non_null),
        dependencia_patrimonial=("dependencia_patrimonial", first_non_null),
        municipio=("municipio", first_non_null),
        tipo_via=("tipo_via", first_non_null),
        nombre_via=("nombre_via", first_non_null),
        numero_via=("numero_via", first_non_null),
        info_adicional=("info_adicional", first_non_null),
        codigo_postal=("codigo_postal", first_non_null),
        utm_x=("utm_x", first_non_null),
        utm_y=("utm_y", first_non_null),
        lon=("lon", first_non_null),
        lat=("lat", first_non_null),
        direccion_completa=("direccion_completa", first_non_null),
        especialidades_texto=("especialidad", join_unique),
        num_especialidades=("especialidad", lambda series: series.dropna().nunique()),
        perfiles_atencion=("especialidad", build_attention_profiles),
    )
)

cleaned = cleaned.sort_values(["municipio", "centro_id"], kind="stable").reset_index(drop=True)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(PROCESSED_PATH, sep=";", index=False, encoding="utf-8-sig")

print(f"Filas originales: {len(raw):,}")
print(f"Hospitales únicos: {len(cleaned):,}")
print(f"Hospitales con coordenadas: {cleaned['lat'].notna().sum():,}")
print(f"Especialidades distintas: {df['especialidad'].nunique():,}")
cleaned.head(10)

Filas originales: 50,749
Hospitales únicos: 15,756
Hospitales con coordenadas: 15,596
Especialidades distintas: 107


,centro_id,centro_tipo,dependencia_funcional,dependencia_patrimonial,municipio,tipo_via,nombre_via,numero_via,info_adicional,codigo_postal,utm_x,utm_y,lon,lat,direccion_completa,especialidades_texto,num_especialidades,perfiles_atencion
0,CS12674,Consultorio de atención primaria,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,"Acebeda, La",PLAZA,de San Miguel,1,NaN,28755,447506.0,4548576.0,-3.624993,41.086756,"PLAZA de San Miguel 1 Acebeda, La 28755",Atención sanitaria domiciliaria | Enfermería |...,3,
1,CS10341,Centro Polivalente,Privado No Benéfico,Privados,Ajalvir,CALLE,Segovia,24,NaN,28864,458967.0,4486674.0,-3.484471,40.529789,CALLE Segovia 24 Ajalvir 28864,Medicina general/de familia | Medicina estética,2,
2,CS10845,Centro de reconocimiento,Privado No Benéfico,Privados,Ajalvir,CTRA,Daganzo,6,NaN,28864,459326.0,4486844.0,-3.480243,40.531338,CTRA Daganzo 6 Ajalvir 28864,Medicina general/de familia | Otras unidades a...,2,
3,CS20349,Clínica dental,Privado No Benéfico,Privados,Ajalvir,CTRA,Daganzo,18,LOCAL,28864,459462.0,4487087.0,-3.478653,40.533533,CTRA Daganzo 18 LOCAL Ajalvir 28864,Odontología/Estomatología,1,
4,CS6489,Servicio sanitario integrado en una organizaci...,Privado No Benéfico,Privados,Ajalvir,CTRA,de Torrejón,KM.3.5,NaN,28864,459346.0,4483105.0,-3.479767,40.497655,CTRA de Torrejón KM.3.5 Ajalvir 28864,Medicina general/de familia | Enfermería,2,
5,CS8906,Consultorio de atención primaria,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Ajalvir,CALLE,Fuente,28-30,NaN,28864,459139.0,4487414.0,-3.482488,40.536463,CALLE Fuente 28-30 Ajalvir 28864,Medicina general/de familia | Atención sanitar...,6,pediatria
6,CS9746,Centro Polivalente,Privado No Benéfico,Privados,Ajalvir,CTRA,Daganzo,6,"KM.0,300 BJ LOC",28864,459326.0,4486844.0,-3.480243,40.531338,"CTRA Daganzo 6 KM.0,300 BJ LOC Ajalvir 28864",Fisioterapia | Medicina estética | Cirugía ort...,22,cardiovascular | trauma_ortopedia | obstetrici...
7,SS01218,Servicio sanitario integrado en una organizaci...,Privado No Benéfico,Privados,Ajalvir,CTRA,de Torrejón,KM-5,NaN,28864,459246.0,4486657.0,-3.481176,40.529649,CTRA de Torrejón KM-5 Ajalvir 28864,Odontología/Estomatología | Medicina general/d...,2,
8,CS12605,Consultorio de atención primaria,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Alameda del Valle,PLAZA,de Santa Marina,17,BAJO,28749,428958.0,4529999.0,-3.843668,40.918019,PLAZA de Santa Marina 17 BAJO Alameda del Vall...,Medicina general/de familia | Atención sanitar...,4,
9,CH0044,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Alcalá de Henares,CTRA,de Meco,S/N,NaN,28805,470511.0,4484529.0,-3.348075,40.510956,CTRA de Meco S/N Alcalá de Henares 28805,Recuperación de oocitos | Extracción de órgano...,80,cardiovascular | neurologia | trauma_ortopedia...


## Qué deja listo esta limpieza

El fichero procesado queda en `analisis_datos/data/processed/centros_servicios_establecimientos_sanitarios_limpio.csv`.

Columnas clave para el prototipo:
- `centro_id`: identificador estable del centro
- `lat` y `lon`: coordenadas ya convertidas para geolocalización
- `direccion_completa`: dirección lista para mostrar o depurar
- `especialidades_texto`: lista completa de especialidades del hospital
- `perfiles_atencion`: categorías resumidas útiles para triaje y derivación
- `num_especialidades`: tamaño del catálogo asistencial del hospital